# Sinh prompt bang VLM cuc bo

Sinh `SAA/prompts/generated/mvtec-blind.json` bang Qwen2.5-VL-3B chay tren GPU
Colab. **Khong goi API ngoai.**

## Notebook nay nhe hon han cac notebook khac

Bien the `blind` chi nhan TEN CLASS, khong nhan anh. Nen khong can:

- Cai GroundingDINO / SAM (~5 phut)
- Tai weight ViT-H 2.4 GB (~3 phut)
- Tai dataset MVTec (~10 phut)

Chi can clone repo + `transformers`. Tong khoang **12 phut**.

## Vi sao 3B fp16 chu khong phai 7B 4-bit

7B fp16 la ~15 GB, sat tran 16 GB cua T4, nen phai luong tu hoa - va
`bitsandbytes` tren Colab dang gay. Ban 3B fp16 chiem ~6 GB, lot thoai mai.

Doi lai la bo duoc mot dependency khoi duong tai lap. Greedy decoding tren trong
so co dinh cho ra cung mot file JSON moi lan chay; them luong tu hoa la them mot
thu phai khop thi dieu do moi dung.

## Nguong can vuot

| | `p_f1` |
|---|---|
| P1 general_prompts | 35.80 |
| P3 prompt thu cong | 37.44 |
| **Chon tot nhat moi class** | **39.77** |

Xem `results/phase_a_ladder/README.md`.

## Cai dat (~2 phut)

In [ ]:
%cd /content
!rm -rf /content/Segment-Any-Anomaly
!git clone -q -b dev https://github.com/SyDuc7421/Segment-Any-Anomaly.git
%cd Segment-Any-Anomaly

# Chi can transformers + accelerate. KHONG can bitsandbytes (chay fp16), khong
# can GroundingDINO/SAM (khong chay inference), khong can dataset (blind chi
# nhan ten class).
!pip install -q "transformers>=4.45" accelerate

import importlib.util
print('transformers:', 'OK' if importlib.util.find_spec('transformers') else 'THIEU')
print('torch       :', 'OK' if importlib.util.find_spec('torch') else 'THIEU')
!git log --oneline -1

## Probe: nap model thu (~3 phut, tai ~6 GB)

In [ ]:
# Probe: nap model va sinh mot cau, truoc khi ton 15 phut cho 15 class.
# Lop Auto* tu tra ra lop dung tu config nen khong phai doan ten lop.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map='auto'
).eval()

print('nap duoc:', type(model).__name__)
print('VRAM    :', f'{torch.cuda.max_memory_allocated() / 1e9:.1f} GB')

messages = [{'role': 'user', 'content': [
    {'type': 'text', 'text': 'Reply with exactly this JSON and nothing else: {"ok": true}'}
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
print('tra loi :', processor.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True).strip())

# Giai phong truoc khi chay script - no tu nap lai model.
del model, processor
torch.cuda.empty_cache()

## Sinh prompt (~10 phut)

In [ ]:
%cd /content/Segment-Any-Anomaly
!python tools/gen_prompts.py \
    --dataset mvtec --variant blind \
    --model Qwen/Qwen2.5-VL-3B-Instruct --fp16 \
    --out SAA/prompts/generated/mvtec-blind.json

## Kiem ket qua

In [ ]:
# Kiem file nap duoc, va dem so luot DINO ma no se ton.
import importlib.util, json, os

spec = importlib.util.spec_from_file_location(
    'llm_prompts', '/content/Segment-Any-Anomaly/SAA/prompts/llm_prompts.py')
lp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lp)

PATH = 'SAA/prompts/generated/mvtec-blind.json'
gen = lp.load_prompt_file(PATH)          # validate tung entry
print(f'{len(gen)} class nap duoc\n')

total = 0
for name, s in gen.items():
    n = len(s['defect_prompts'])
    total += n
    print(f'{name:12s} object={s["object_prompt"]:14s} {n} prompt  '
          f'{[p["text"] for p in s["defect_prompts"]]}')

calls = 1 + total / len(gen)
print(f'\ntrung binh {total / len(gen):.1f} prompt/class -> {calls:.2f} luot DINO/anh')
print(f'P3 thu cong dung 5.20 luot. Uoc t_total = {calls * 265 + 530:.0f} ms '
      f'so voi 1926 ms cua P3 -> {1926 / (calls * 265 + 530):.2f}x')

## Kiem nhiem du lieu huan luyen

In [ ]:
# Kiem nhiem du lieu huan luyen.
#
# SAA+ la repo cong khai, paper dang IEEE. VLM co the da thay prompt cua tac gia
# trong du lieu huan luyen. Neu P2 tien sat P3 mot cach dang ngo, do co the la
# NHO chu khong phai suy. Con so nay vao phan Limitations cua luan van bat ke
# ket qua the nao - day la cau hoi hoi dong se hoi.
import sys
sys.path.insert(0, '/content/Segment-Any-Anomaly')
from SAA.prompts.mvtec_parameters import manual_prompts

exact = total_llm = 0
for cls, manual in manual_prompts.items():
    if cls not in gen:
        continue
    manual_texts = {p[0].strip().lower().rstrip('.') for p in manual}
    llm_texts = {p['text'].strip().lower().rstrip('.') for p in gen[cls]['defect_prompts']}
    overlap = manual_texts & llm_texts
    total_llm += len(llm_texts)
    exact += len(overlap)
    if overlap:
        print(f'{cls:12s} trung khit: {sorted(overlap)}')

print(f'\n{exact}/{total_llm} prompt trung khit tung chu voi prompt thu cong')
if exact == 0:
    print('Khong trung chu nao -> khong co dau hieu nho truc tiep.')
else:
    print('Ty le cao tren nhieu class la dau hieu nho. Trung y ma khac chu la suy that.')

## Luu ra Drive

In [ ]:
# Luu ra Drive. /content mat khi runtime bi kill, va file nay phai duoc COMMIT
# vao git - do la dieu kien de hoi dong tai lap ma khong can chay lai VLM.
from google.colab import drive
import os, shutil

try:
    drive.mount('/content/drive')
except ValueError:
    drive.mount('/content/drive', force_remount=True)

dst = '/content/drive/MyDrive/SAA_results/generated_prompts'
os.makedirs(dst, exist_ok=True)

for name in ('mvtec-blind.json', 'mvtec-blind-meta.json'):
    src = f'/content/Segment-Any-Anomaly/SAA/prompts/generated/{name}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'{os.path.getsize(src):>8,} B  {name}')

print(f'\n-> {dst}')
print('\nTai hai file nay ve may, dat vao SAA/prompts/generated/ roi commit.')
print('File -meta.json ghi model, decoding, va nguyen van prompt da dung.')